# 병렬 섹셔닝과 투표

독립적인 섹션과 평가 호출을 `asyncio.gather`로 함께 기다립니다.

## 실행 환경과 비용 상한

최대 8회, 호출당 출력 220토큰으로 제한합니다. 계획 비용은 0.0199달러 미만입니다.

## 코드를 읽기 전에

`AsyncOpenAI`는 비동기 클라이언트이며 `await client.responses.create(...)`에서 각 네트워크 응답을 기다립니다. `asyncio.gather(*jobs)`는 여러 코루틴을 함께 예약하고 모든 결과가 끝난 뒤 입력 순서의 목록을 반환합니다. 요청을 동시에 시작할 수 있다는 뜻이지 제공자 내부 처리 시간이나 완료 순서가 같다는 뜻은 아닙니다.

섹셔닝은 서로 의존하지 않는 목적·절차·지원 문단을 나눕니다. 투표는 같은 두 후보를 여러 평가 기준에 보냅니다. 전자는 결과를 합쳐 한 문서를 만들고, 후자는 판정 분포를 남긴다는 점에서 출력의 합치는 방식이 다릅니다.

In [1]:
# 이 장은 OpenAI SDK를 직접 배우므로 두 패키지만 고정합니다.
%pip install -q openai==2.29.0 python-dotenv==1.2.3


Note: you may need to restart the kernel to use updated packages.


### 다음 셀을 읽는 방법

`load_dotenv(find_dotenv(usecwd=True))`는 현재 폴더에서 위로 올라가며 `.env`를 찾아 환경 변수에 넣습니다. `AsyncOpenAI(timeout=45.0, max_retries=0)`는 비동기 클라이언트를 만들며 아직 모델을 호출하지 않습니다. `calls`는 이후 성공 응답의 지연과 토큰 수를 모을 빈 목록입니다.


In [2]:
# 비동기 API 호출과 사용량 측정에 필요한 도구를 준비합니다.
import asyncio
import json
import os
import re
import time
from collections import Counter
from dotenv import find_dotenv, load_dotenv
from openai import AsyncOpenAI
load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "프로젝트 루트 .env에 OPENAI_API_KEY가 필요합니다."
MODEL = "gpt-5.4-mini"
client = AsyncOpenAI(timeout=45.0, max_retries=0)
calls: list[dict[str, float | int | str]] = []
print("모델:", MODEL, "| API 키 존재: 예")


모델: gpt-5.4-mini | API 키 존재: 예


### 다음 셀을 읽는 방법

`call_json(instructions, prompt, limit)`은 지시문·사용자 입력·최대 출력 토큰을 받는 코루틴입니다. `await client.responses.create()`에서 실제 요청을 기다리고, 응답의 JSON 텍스트를 파이썬 사전으로 바꾸어 반환합니다. `calls`에는 성공 응답의 지연과 입력·출력 토큰만 추가됩니다.


In [3]:
# 비동기 Responses API의 JSON 응답과 사용량을 기록합니다.
async def call_json(instructions: str, prompt: str, limit: int = 220) -> dict:
    started = time.perf_counter()
    response = await client.responses.create(
        model=MODEL,
        instructions=instructions + " 반드시 유효한 JSON 객체만 반환하세요.",
        input="JSON 요청\n" + prompt,
        max_output_tokens=limit,
        reasoning={"effort": "none"},
        text={"format": {"type": "json_object"}},
    )
    calls.append({"latency_s": time.perf_counter() - started,
                  "input": response.usage.input_tokens,
                  "output": response.usage.output_tokens})
    return json.loads(response.output_text)


### 다음 셀을 읽는 방법

`call_answer()`는 `answer` 문자열 하나만 허용하는 JSON Schema를 요청에 넣는 코루틴입니다. 비동기 SDK 응답을 사전으로 바꾼 뒤 값이 문자열이며 비어 있지 않은지 다시 검사하고 사전을 반환합니다. 스키마 통과는 답의 사실 정확성까지 보장하지 않습니다.


In [4]:
# 비동기 응답의 answer 형식을 엄격하게 검사합니다.
async def call_answer(instructions: str, prompt: str, limit: int = 400) -> dict:
    started = time.perf_counter()
    response = await client.responses.create(
        model=MODEL, instructions=instructions, input=prompt,
        max_output_tokens=limit, reasoning={"effort": "none"},
        text={"format": {"type": "json_schema", "name": "answer_payload",
              "schema": {"type": "object", "properties": {"answer": {"type": "string"}},
              "required": ["answer"], "additionalProperties": False}, "strict": True}},
    )
    calls.append({"latency_s": time.perf_counter() - started,
                  "input": response.usage.input_tokens, "output": response.usage.output_tokens})
    data = json.loads(response.output_text)
    if not isinstance(data.get("answer"), str) or not data["answer"].strip():
        raise ValueError("응답의 answer가 비어 있습니다.")
    return data


### 다음 셀을 읽는 방법

`cost_usd(records)`는 앞에서 모은 성공 응답의 입력·출력 토큰을 합산해 이 실습에 고정한 단가를 적용합니다. `score_of(data)`는 평가 점수가 불리언이 아닌 숫자이며 0~100 범위인지 확인한 뒤 정수로 반환합니다. 두 함수 모두 모델을 호출하지 않습니다.


In [5]:
# 반환된 토큰 사용량으로 할인 없는 비용을 추정합니다.
def cost_usd(records: list[dict]) -> float:
    return (sum(int(r["input"]) for r in records) * 0.75
            + sum(int(r["output"]) for r in records) * 4.50) / 1_000_000
def score_of(data: dict) -> int:
    value = data.get("score")
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        raise ValueError("평가 응답의 score는 숫자여야 합니다.")
    if not 0 <= value <= 100:
        raise ValueError("평가 응답의 score는 0~100이어야 합니다.")
    return int(value)


## 섹셔닝: 서로 의존하지 않는 세 부분을 병렬 작성

In [6]:
# 독립적인 세 섹션을 동시에 작성하고 지연을 측정합니다.
brief = "2026년 10월 15일부터 사내 2단계 인증을 적용합니다. 대상은 전 직원입니다."
sections = [
    ("목적", "도입 목적을 2문장으로"),
    ("절차", "등록 절차를 번호 3개로"),
    ("지원", "주의점과 문의 방법을 2문장으로"),
]
async def write_section(name: str, request: str) -> dict:
    prompt = f"공통 사실: {brief}\n부분: {name}\n요청: {request}"
    result = await call_json("사내 안내문 작성자입니다. name과 text를 반환합니다.", prompt)
    text = result.get("text")
    if not isinstance(text, str) or not text.strip():
        raise ValueError(f"{name} 섹션의 text가 비어 있습니다.")
    return {"name": name, "text": text.strip()}
started = time.perf_counter()
parts = await asyncio.gather(*(write_section(*item) for item in sections))
wall = time.perf_counter() - started
print("입력 순서대로 반환:", [part["name"] for part in parts])
print("병렬 벽시계 시간(초):", round(wall, 3))
print("개별 지연 합(초):", round(sum(float(r["latency_s"]) for r in calls), 3))


입력 순서대로 반환: ['목적', '절차', '지원']
병렬 벽시계 시간(초): 2.283
개별 지연 합(초): 5.644


## 투표: 후보와 평가자를 병렬 실행

In [7]:
# 두 후보를 만들고 세 평가자의 표를 수집한 뒤 유효표를 집계합니다.
candidate_prompts = ["정중하고 자세한 안내", "짧고 행동 중심인 안내"]
candidates = await asyncio.gather(*[
    call_answer("사내 안내문 작성자입니다.",
                f"{brief} {style}를 4문장으로 작성하세요.", 200)
    for style in candidate_prompts
])
async def vote(judge: str) -> dict:
    prompt = f"A={candidates[0]['answer']}\nB={candidates[1]['answer']}"
    return await call_json(
        f"{judge} 기준 평가자입니다. winner는 A, B, tie, refuse 중 하나입니다.",
        prompt + "\n사실 정확성과 실행 가능성을 평가하세요.", 120)
votes = await asyncio.gather(*(vote(name) for name in ["인사", "보안", "신입 직원"]))
valid_votes = [v.get("winner") for v in votes if v.get("winner") in {"A", "B"}]
counts = Counter(valid_votes)
if not valid_votes:
    winner = "undecided"
elif counts.get("A", 0) == counts.get("B", 0):
    winner = "tie"
else:
    winner = counts.most_common(1)[0][0]
print("원표:", votes)
print("유효표:", dict(counts), "| 결과:", winner)
print("승자 집계 제외 표:", len(votes) - len(valid_votes))


원표: [{'winner': 'A'}, {'winner': 'B'}, {'winner': 'A'}]
유효표: {'A': 2, 'B': 1} | 결과: A
승자 집계 제외 표: 0


## 실험 결과 해석

`gather`는 입력 순서대로 결과 목록을 반환하지만 각 요청의 완료 시각까지 같게 만들지는 않습니다. 벽시계 시간은 네트워크 상태와 서비스 부하에 따라 달라집니다. 투표가 동점이거나 평가자가 거절하면 결과를 그대로 기록하며 임의로 승자를 만들지 않습니다.

In [8]:
# 병렬 실험의 호출 수와 토큰, 비용 추정치를 출력합니다.
print("총 호출:", len(calls))
print("입력 토큰:", sum(int(r["input"]) for r in calls))
print("출력 토큰:", sum(int(r["output"]) for r in calls))
print("사용량 기준 추정 비용($):", round(cost_usd(calls), 6))


총 호출: 8
입력 토큰: 1251
출력 토큰: 546
사용량 기준 추정 비용($): 0.003395


## 다른 상황에 적용하기

한 보고서의 매출 표와 고객 인터뷰 요약은 동시에 만들 수 있지만, 두 결과를 인용하는 결론은 먼저 실행할 수 없습니다. 보안 검토에서는 같은 코드에 서로 다른 취약점 관점을 병렬 적용할 수 있습니다. 병렬화는 의존성이 없는 구간의 대기 시간을 겹치게 할 뿐이며, 호출 수와 오류 처리 책임을 줄여 주지는 않습니다.